# Important note on the HMM information gap

The final manuscript does **not** use an information gap obtained by subtracting independently estimated Monte Carlo cross-entropies from this original extension workflow. That estimator can produce small negative values from sampling noise. The authoritative final information-gap analysis uses matched/same-history quantities implemented in `HMM_Information_Gap_Correction_GPU.ipynb` and archived in `HMM_Oracle_Gap_Fix_GPU_RESULTS.zip`.

This notebook is retained for experimental provenance of the latent-state extension; use the corrected notebook/result archive for the reported finite-context information gap.


# HMM Latent-State Extension

Controlled latent-state extension of the context-sufficiency framework. The experiment compares exact Bayesian finite-context prediction, scratch-Transformer performance, latent-posterior accessibility, and predictive readout across context lengths.

Unlike the finite-order Markov experiments, this HMM generally has no exact finite context boundary.


In [ ]:
import os, sys, json, math, random, gc, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE=Path.cwd()/"Markov_Context_Paper"
OUT=BASE/"HMM_Latent_State_Extension"
RUN_DIR=OUT/"runs"
CSV_DIR=OUT/"csv"
FIG_DIR=OUT/"figures"
CKPT_DIR=OUT/"checkpoints"
for p in [OUT,RUN_DIR,CSV_DIR,FIG_DIR,CKPT_DIR]:
    p.mkdir(parents=True,exist_ok=True)

V=8
KZ=8
SEEDS=[0,1,2]
CONTEXT_LENGTHS=[2,4,8,16,32]

TRANSITION_STICKINESS=5.0
EMISSION_CONCENTRATION=0.55

D_MODEL=256
NHEAD=8
NLAYERS=3
DIM_FF=512
DROPOUT=0.0

TRAIN_STEPS=2500
BATCH_SIZE=512
LR=3e-4
WEIGHT_DECAY=1e-4
EVAL_EVERY=250
EVAL_BATCHES=20

N_EVAL=30000
N_PROBE_TRAIN=20000
N_PROBE_VAL=5000
N_PROBE_TEST=10000
PROBE_BATCH=1024
PROBE_EPOCHS=30
PROBE_PATIENCE=5

LONG_HISTORY=256

print("Device:",DEVICE)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
print("Output:",OUT)


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def clean():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def save_json(path,obj):
    path.parent.mkdir(parents=True,exist_ok=True)
    with open(path,"w",encoding="utf-8") as f:
        json.dump(obj,f,indent=2,default=lambda x:
                  float(x) if isinstance(x,np.floating) else
                  int(x) if isinstance(x,np.integer) else str(x))

save_json(OUT/"config.json",{
    "V":V,"KZ":KZ,"SEEDS":SEEDS,"CONTEXT_LENGTHS":CONTEXT_LENGTHS,
    "TRANSITION_STICKINESS":TRANSITION_STICKINESS,
    "EMISSION_CONCENTRATION":EMISSION_CONCENTRATION,
    "D_MODEL":D_MODEL,"NHEAD":NHEAD,"NLAYERS":NLAYERS,"DIM_FF":DIM_FF,
    "TRAIN_STEPS":TRAIN_STEPS,"BATCH_SIZE":BATCH_SIZE,"LR":LR,
    "N_EVAL":N_EVAL,"N_PROBE_TRAIN":N_PROBE_TRAIN,
    "N_PROBE_VAL":N_PROBE_VAL,"N_PROBE_TEST":N_PROBE_TEST,
    "LONG_HISTORY":LONG_HISTORY,"torch":torch.__version__,
    "python":sys.version,"device":str(DEVICE)
})


## HMM construction

Each seed defines an 8-state latent Markov chain with moderately sticky transitions and stochastic state-specific emissions over 8 observed symbols.

In [ ]:
def make_hmm(seed,V=V,K=KZ):
    rng=np.random.default_rng(seed+91337)

    raw_T=rng.gamma(shape=1.0,scale=1.0,size=(K,K))
    raw_T += TRANSITION_STICKINESS*np.eye(K)
    T=raw_T/raw_T.sum(axis=1,keepdims=True)

    raw_E=rng.gamma(shape=EMISSION_CONCENTRATION,scale=1.0,size=(K,V))
    E=raw_E/raw_E.sum(axis=1,keepdims=True)

    pi=np.full(K,1.0/K)
    for _ in range(100000):
        new=pi@T
        if np.max(np.abs(new-pi))<1e-14:
            pi=new
            break
        pi=new
    pi=pi/pi.sum()

    return (
        torch.tensor(T,dtype=torch.float64),
        torch.tensor(E,dtype=torch.float64),
        torch.tensor(pi,dtype=torch.float64),
    )

for seed in SEEDS:
    T,E,pi=make_hmm(seed)
    print("seed",seed,"mean Tdiag",float(torch.diag(T).mean()))


## Exact Bayesian finite-context oracle

In [ ]:
@torch.no_grad()
def filter_contexts(x,T,E,pi):
    x=x.cpu()
    B,L=x.shape
    post=pi[None,:].repeat(B,1).clone()

    for t in range(L):
        obs=x[:,t]
        likelihood=E[:,obs].T
        post=post*likelihood
        post=post/post.sum(dim=1,keepdim=True).clamp_min(1e-300)

        if t<L-1:
            post=post@T

    latent_next=post@T
    pred=latent_next@E
    pred=pred/pred.sum(dim=1,keepdim=True).clamp_min(1e-300)
    return post,pred

@torch.no_grad()
def oracle_ce(x,y,T,E,pi):
    _,pred=filter_contexts(x,T,E,pi)
    p=pred[torch.arange(len(y)),y.cpu()].clamp_min(1e-300)
    return float((-torch.log(p)).mean())

@torch.no_grad()
def posterior_entropy(post):
    return -(post*torch.log(post.clamp_min(1e-300))).sum(-1)


## HMM sampler

In [ ]:
@torch.no_grad()
def sample_hmm_sequences(T,E,pi,seq_len,batch_size,seed):
    g=torch.Generator(device="cpu")
    g.manual_seed(seed)

    B=batch_size
    z=torch.empty((B,seq_len),dtype=torch.long)
    x=torch.empty((B,seq_len),dtype=torch.long)

    z[:,0]=torch.multinomial(pi.float(),B,replacement=True,generator=g)
    x[:,0]=torch.multinomial(E[z[:,0]].float(),1,generator=g).squeeze(1)

    for t in range(1,seq_len):
        z[:,t]=torch.multinomial(T[z[:,t-1]].float(),1,generator=g).squeeze(1)
        x[:,t]=torch.multinomial(E[z[:,t]].float(),1,generator=g).squeeze(1)

    return x,z


## Scratch Transformer

In [ ]:
class TinyCausalTransformer(nn.Module):
    def __init__(self,vocab_size=V,d_model=D_MODEL,nhead=NHEAD,
                 num_layers=NLAYERS,dim_ff=DIM_FF,max_len=max(CONTEXT_LENGTHS)):
        super().__init__()
        self.tok=nn.Embedding(vocab_size,d_model)
        self.pos=nn.Embedding(max_len,d_model)

        layer=nn.TransformerEncoderLayer(
            d_model=d_model,nhead=nhead,dim_feedforward=dim_ff,
            dropout=DROPOUT,activation="gelu",batch_first=True,norm_first=True
        )
        self.enc=nn.TransformerEncoder(layer,num_layers=num_layers)
        self.ln=nn.LayerNorm(d_model)
        self.head=nn.Linear(d_model,vocab_size)

    def forward(self,x,return_hidden=False):
        B,L=x.shape
        pos=torch.arange(L,device=x.device)[None,:]
        h=self.tok(x)+self.pos(pos)

        mask=torch.full((L,L),float("-inf"),device=x.device)
        mask=torch.triu(mask,diagonal=1)

        h=self.enc(h,mask=mask)
        h=self.ln(h)
        logits=self.head(h)

        if return_hidden:
            return logits,h
        return logits


In [ ]:
@torch.no_grad()
def eval_model(model,T,E,pi,L,batches=EVAL_BATCHES,batch_size=BATCH_SIZE,seed=999):
    model.eval()
    total=0.0
    n=0
    for b in range(batches):
        x,_=sample_hmm_sequences(T,E,pi,L+1,batch_size,seed+b)
        inp=x[:,:L].to(DEVICE)
        y=x[:,L].to(DEVICE)
        logits=model(inp)[:,-1,:]
        ce=F.cross_entropy(logits,y)
        total += ce.item()*len(y)
        n += len(y)
    return total/n

def train_one_model(seed,L,T,E,pi):
    ckpt=CKPT_DIR/f"hmm_seed{seed}_L{L}.pt"
    meta=RUN_DIR/f"hmm_seed{seed}_L{L}.json"

    model=TinyCausalTransformer().to(DEVICE)

    if ckpt.exists() and meta.exists():
        state=torch.load(ckpt,map_location=DEVICE)
        model.load_state_dict(state["model"])
        return model,json.loads(meta.read_text())

    set_seed(10000+100*seed+L)
    model=TinyCausalTransformer().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)

    best_val=float("inf")
    best_state=None
    history=[]

    for step in range(1,TRAIN_STEPS+1):
        model.train()
        x,_=sample_hmm_sequences(
            T,E,pi,L+1,BATCH_SIZE,
            seed=1000000+seed*100000+L*10000+step
        )
        inp=x[:,:L].to(DEVICE)
        y=x[:,L].to(DEVICE)

        opt.zero_grad(set_to_none=True)
        logits=model(inp)[:,-1,:]
        loss=F.cross_entropy(logits,y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()

        if step%EVAL_EVERY==0 or step==TRAIN_STEPS:
            val_ce=eval_model(
                model,T,E,pi,L,
                batches=EVAL_BATCHES,
                seed=2000000+seed*10000+L*100
            )
            history.append({"step":step,"train_ce":float(loss.item()),"val_ce":val_ce})
            print(f"seed={seed} L={L} step={step} train={loss.item():.4f} val={val_ce:.4f}")

            if val_ce<best_val:
                best_val=val_ce
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    torch.save({"model":model.state_dict()},ckpt)
    obj={"seed":seed,"L":L,"best_val_ce":best_val,"history":history}
    save_json(meta,obj)
    return model,obj


## Native model performance vs exact oracle

In [ ]:
native_path=CSV_DIR/"HMM_native_oracle_results.csv"

if native_path.exists():
    native_df=pd.read_csv(native_path)
    print("Loaded",native_path)
else:
    rows=[]
    for seed in SEEDS:
        T,E,pi=make_hmm(seed)

        x_long,_=sample_hmm_sequences(
            T,E,pi,LONG_HISTORY+1,N_EVAL,seed=8000000+seed
        )
        H_long=oracle_ce(x_long[:,:LONG_HISTORY],x_long[:,LONG_HISTORY],T,E,pi)

        for L in CONTEXT_LENGTHS:
            print("Native/oracle seed",seed,"L",L)
            model,meta=train_one_model(seed,L,T,E,pi)

            x,_=sample_hmm_sequences(
                T,E,pi,L+1,N_EVAL,seed=9000000+seed*1000+L
            )
            context=x[:,:L]
            y=x[:,L]

            post,pred=filter_contexts(context,T,E,pi)
            H_L=float((-torch.log(pred[torch.arange(N_EVAL),y].clamp_min(1e-300))).mean())
            post_H=float(posterior_entropy(post).mean())

            model.eval()
            total=0.0
            n=0
            with torch.no_grad():
                for s in range(0,N_EVAL,2048):
                    xb=context[s:s+2048].to(DEVICE)
                    yb=y[s:s+2048].to(DEVICE)
                    logits=model(xb)[:,-1,:]
                    total += F.cross_entropy(logits,yb).item()*len(yb)
                    n += len(yb)
            model_ce=total/n

            rows.append({
                "seed":seed,
                "L":L,
                "oracle_ce_HL":H_L,
                "long_history_oracle_ce":H_long,
                "information_gap_to_long":H_L-H_long,
                "model_ce":model_ce,
                "model_excess_over_HL":model_ce-H_L,
                "posterior_entropy_z":post_H,
                "best_val_ce":meta["best_val_ce"],
            })

            del model
            clean()

    native_df=pd.DataFrame(rows)
    native_df.to_csv(native_path,index=False)

display(native_df)


## Representation and predictive-readout probes

In [ ]:
class PosteriorProbe(nn.Module):
    def __init__(self,d=D_MODEL,k=KZ):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d,256),nn.GELU(),nn.Linear(256,k))
    def forward(self,x):
        return self.net(x)

class NextTokenProbe(nn.Module):
    def __init__(self,d=D_MODEL,v=V):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d,256),nn.GELU(),nn.Linear(256,v))
    def forward(self,x):
        return self.net(x)

@torch.no_grad()
def extract_hidden_and_targets(model,T,E,pi,L,n_total,seed):
    x,_=sample_hmm_sequences(T,E,pi,L+1,n_total,seed)
    context=x[:,:L]
    y=x[:,L]
    post,pred=filter_contexts(context,T,E,pi)

    hs=[]
    model.eval()
    for s in range(0,n_total,1024):
        xb=context[s:s+1024].to(DEVICE)
        _,h=model(xb,return_hidden=True)
        hs.append(h[:,-1,:].cpu())

    return torch.cat(hs),post.float(),pred.float(),y

def train_posterior_probe(Xtr,Ptr,Xva,Pva):
    model=PosteriorProbe().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-4)

    tr=DataLoader(TensorDataset(Xtr,Ptr),batch_size=PROBE_BATCH,shuffle=True)
    va=DataLoader(TensorDataset(Xva,Pva),batch_size=PROBE_BATCH)

    best=float("inf")
    best_state=None
    bad=0

    for ep in range(PROBE_EPOCHS):
        model.train()
        for xb,pb in tr:
            xb,pb=xb.to(DEVICE),pb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logq=F.log_softmax(model(xb),dim=-1)
            loss=F.kl_div(logq,pb,reduction="batchmean")
            loss.backward()
            opt.step()

        model.eval()
        tot=0.0
        n=0
        with torch.no_grad():
            for xb,pb in va:
                xb,pb=xb.to(DEVICE),pb.to(DEVICE)
                logq=F.log_softmax(model(xb),dim=-1)
                loss=F.kl_div(logq,pb,reduction="batchmean")
                tot += loss.item()*len(xb)
                n += len(xb)
        v=tot/n

        if v<best-1e-5:
            best=v
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad += 1
            if bad>=PROBE_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def train_next_probe(Xtr,ytr,Xva,yva):
    model=NextTokenProbe().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-4)

    tr=DataLoader(TensorDataset(Xtr,ytr),batch_size=PROBE_BATCH,shuffle=True)
    va=DataLoader(TensorDataset(Xva,yva),batch_size=PROBE_BATCH)

    best=float("inf")
    best_state=None
    bad=0

    for ep in range(PROBE_EPOCHS):
        model.train()
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=F.cross_entropy(model(xb),yb)
            loss.backward()
            opt.step()

        model.eval()
        tot=0.0
        n=0
        with torch.no_grad():
            for xb,yb in va:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE)
                loss=F.cross_entropy(model(xb),yb)
                tot += loss.item()*len(xb)
                n += len(xb)
        v=tot/n

        if v<best-1e-5:
            best=v
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad += 1
            if bad>=PROBE_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


In [ ]:
probe_path=CSV_DIR/"HMM_probe_results.csv"

if probe_path.exists():
    probe_df=pd.read_csv(probe_path)
    print("Loaded",probe_path)
else:
    rows=[]
    n_total=N_PROBE_TRAIN+N_PROBE_VAL+N_PROBE_TEST
    a=slice(0,N_PROBE_TRAIN)
    b=slice(N_PROBE_TRAIN,N_PROBE_TRAIN+N_PROBE_VAL)
    c=slice(N_PROBE_TRAIN+N_PROBE_VAL,n_total)

    for seed in SEEDS:
        T,E,pi=make_hmm(seed)

        for L in CONTEXT_LENGTHS:
            print("Probes seed",seed,"L",L)
            model,_=train_one_model(seed,L,T,E,pi)

            X,Pz,Pnext,y=extract_hidden_and_targets(
                model,T,E,pi,L,n_total,
                seed=12000000+seed*1000+L
            )

            pprobe=train_posterior_probe(X[a],Pz[a],X[b],Pz[b])
            pprobe.eval()
            logits=[]
            with torch.no_grad():
                for s in range(0,len(X[c]),2048):
                    logits.append(pprobe(X[c][s:s+2048].to(DEVICE)).cpu())
            logits=torch.cat(logits)
            Q=torch.softmax(logits,dim=-1)
            P=Pz[c]

            kl=float((P*(torch.log(P.clamp_min(1e-9))-torch.log(Q.clamp_min(1e-9)))).sum(-1).mean())
            l1=float(torch.abs(P-Q).sum(-1).mean())
            arg_acc=float((P.argmax(-1)==Q.argmax(-1)).float().mean())

            nprobe=train_next_probe(X[a],y[a],X[b],y[b])
            nprobe.eval()

            total=0.0
            n=0
            with torch.no_grad():
                for s in range(0,len(X[c]),2048):
                    xb=X[c][s:s+2048].to(DEVICE)
                    yb=y[c][s:s+2048].to(DEVICE)
                    z=nprobe(xb)
                    total += F.cross_entropy(z,yb).item()*len(yb)
                    n += len(yb)
            readout_ce=total/n

            Pt=Pnext[c]
            yt=y[c]
            oracle_test_ce=float(
                (-torch.log(Pt[torch.arange(len(yt)),yt].clamp_min(1e-9))).mean()
            )

            rows.append({
                "seed":seed,
                "L":L,
                "posterior_probe_KL":kl,
                "posterior_probe_L1":l1,
                "posterior_argmax_acc":arg_acc,
                "readout_ce":readout_ce,
                "oracle_test_ce":oracle_test_ce,
                "readout_excess_over_oracle":readout_ce-oracle_test_ce,
            })

            del model,pprobe,nprobe,X,Pz,Pnext,y
            clean()

    probe_df=pd.DataFrame(rows)
    probe_df.to_csv(probe_path,index=False)

display(probe_df)


## Summary tables

In [ ]:
native_summary=native_df.groupby("L",as_index=False).agg(
    oracle_ce_mean=("oracle_ce_HL","mean"),
    oracle_ce_sd=("oracle_ce_HL","std"),
    long_oracle_mean=("long_history_oracle_ce","mean"),
    info_gap_mean=("information_gap_to_long","mean"),
    info_gap_sd=("information_gap_to_long","std"),
    model_ce_mean=("model_ce","mean"),
    model_ce_sd=("model_ce","std"),
    model_excess_mean=("model_excess_over_HL","mean"),
    model_excess_sd=("model_excess_over_HL","std"),
    posterior_entropy_mean=("posterior_entropy_z","mean"),
)
native_summary.to_csv(CSV_DIR/"HMM_native_oracle_summary.csv",index=False)

probe_summary=probe_df.groupby("L",as_index=False).agg(
    posterior_KL_mean=("posterior_probe_KL","mean"),
    posterior_KL_sd=("posterior_probe_KL","std"),
    posterior_L1_mean=("posterior_probe_L1","mean"),
    posterior_argmax_acc_mean=("posterior_argmax_acc","mean"),
    readout_ce_mean=("readout_ce","mean"),
    readout_ce_sd=("readout_ce","std"),
    readout_excess_mean=("readout_excess_over_oracle","mean"),
    readout_excess_sd=("readout_excess_over_oracle","std"),
)
probe_summary.to_csv(CSV_DIR/"HMM_probe_summary.csv",index=False)

display(native_summary)
display(probe_summary)


In [ ]:
fig,ax=plt.subplots(figsize=(7.5,4.8))
ax.plot(native_summary.L,native_summary.oracle_ce_mean,marker="o",label="Exact finite-context oracle")
ax.plot(native_summary.L,native_summary.model_ce_mean,marker="o",label="Scratch Transformer")
ax.axhline(native_summary.long_oracle_mean.mean(),linestyle="--",label="Long-history oracle")
ax.set_xscale("log",base=2)
ax.set_xlabel("Context length L")
ax.set_ylabel("Cross-entropy (nats)")
ax.set_title("HMM predictive performance vs exact Bayesian oracle")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR/"HMM_oracle_vs_model_ce.png",dpi=300)
plt.show()

fig,ax=plt.subplots(figsize=(7.5,4.8))
ax.plot(native_summary.L,native_summary.info_gap_mean,marker="o",label="Finite-context information gap")
ax.plot(native_summary.L,native_summary.model_excess_mean,marker="o",label="Model excess over H_L")
ax.set_xscale("log",base=2)
ax.set_xlabel("Context length L")
ax.set_ylabel("Excess CE (nats)")
ax.set_title("Information limitation vs model/optimization gap")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR/"HMM_gap_decomposition.png",dpi=300)
plt.show()

fig,ax=plt.subplots(figsize=(7.5,4.8))
ax.plot(probe_summary.L,probe_summary.posterior_KL_mean,marker="o",label="Latent posterior probe KL")
ax.plot(probe_summary.L,probe_summary.readout_excess_mean,marker="o",label="Predictive readout excess CE")
ax.set_xscale("log",base=2)
ax.set_xlabel("Context length L")
ax.set_ylabel("Gap (nats)")
ax.set_title("Latent-state accessibility and predictive readout")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR/"HMM_representation_readout.png",dpi=300)
plt.show()


## Audit and ZIP export

In [ ]:
expected=len(SEEDS)*len(CONTEXT_LENGTHS)
assert len(native_df)==expected,(len(native_df),expected)
assert len(probe_df)==expected,(len(probe_df),expected)
assert np.isfinite(native_df.select_dtypes(include=[np.number]).to_numpy()).all()
assert np.isfinite(probe_df.select_dtypes(include=[np.number]).to_numpy()).all()

audit={
    "status":"PASS",
    "native_rows":len(native_df),
    "probe_rows":len(probe_df),
    "seeds":SEEDS,
    "context_lengths":CONTEXT_LENGTHS,
    "all_numeric_finite":True,
    "note":"Controlled latent-state generalization study; no exact finite context boundary is asserted."
}
save_json(OUT/"HMM_audit.json",audit)

readme=(
    "HMM Latent-State Extension\n\n"
    "Main outputs:\n"
    "- csv/HMM_native_oracle_results.csv\n"
    "- csv/HMM_native_oracle_summary.csv\n"
    "- csv/HMM_probe_results.csv\n"
    "- csv/HMM_probe_summary.csv\n"
    "- figures/HMM_oracle_vs_model_ce.png\n"
    "- figures/HMM_gap_decomposition.png\n"
    "- figures/HMM_representation_readout.png\n"
    "- HMM_audit.json\n\n"
    "Guardrails:\n"
    "1. No L=k theorem is claimed for the HMM.\n"
    "2. Finite-window Bayesian filtering defines the exact H_L oracle.\n"
    "3. Long-history filtering is a practical reference.\n"
    "4. Probe decodability does not imply native causal use.\n"
)
(OUT/"README.txt").write_text(readme,encoding="utf-8")

zip_path=Path.cwd()/"HMM_Latent_State_Extension_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f,f.relative_to(OUT.parent))

print(json.dumps(audit,indent=2))
print("Created:",zip_path)
